# PharmaSpot Morocco — run the API in Google Colab

This notebook **installs the backend**, starts **FastAPI**, and gives you a **public HTTPS link** (via ngrok) so you can point **Vercel** at it with `VITE_API_BASE_URL`.

**Limits (be honest):**
- Colab **disconnects** when idle or after a time limit; the link **stops working** until you re-run cells.
- Free ngrok URLs **change each session** unless you add an [ngrok auth token](https://dashboard.ngrok.com/get-started/your-authtoken).
- First analysis can be **slow** (Overpass + WorldPop); heavy cities may **run out of RAM** on free Colab.
- The **React map UI** is not in this notebook — deploy the `frontend/` on **Vercel** and set `VITE_API_BASE_URL` to the ngrok URL below.


## 1) Clone the project

Change `REPO` if you use a fork.

In [ ]:
import shutil
import subprocess

REPO = "https://github.com/adilchetouani-rgb/pharmacie.git"
BRANCH = "main"
DEST = "/content/pharmacie"

# Colab sometimes fails shallow clone (exit 128); full clone is more reliable.
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "git"], check=False)

shutil.rmtree(DEST, ignore_errors=True)

def _clone(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p.returncode, p.stdout, p.stderr

code, out, err = _clone(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST]
)
if code != 0:
    print("Shallow clone stderr:", err or out)
    shutil.rmtree(DEST, ignore_errors=True)
    code, out, err = _clone(["git", "clone", REPO, DEST])
    if code != 0:
        raise RuntimeError(f"git clone failed:\n{err}\n{out}")
    subprocess.run(["git", "-C", DEST, "checkout", BRANCH], check=True)

print("Clone OK.")
%cd /content/pharmacie

## 2) System libraries (GDAL / spatialindex) + Python packages

This can take **several minutes**.

In [ ]:
%%capture cap --no-stderr
!apt-get update -qq
!apt-get install -y -qq gdal-bin libgdal-dev libspatialindex-dev g++
!pip install -q -U pyngrok
!pip install -q -r /content/pharmacie/backend/requirements.txt

## 3) ngrok authtoken (required)

Since 2024, **anonymous tunnels are blocked**. You must:

1. Sign up at [ngrok](https://dashboard.ngrok.com/signup) and **verify your email**.
2. Open [Your authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) and copy the token.
3. In Colab: **Secrets** (key icon) → **Add secret** → **Name:** exactly `NGROK_AUTHTOKEN` → **Value:** paste **only the token** (one long string). **Do not** paste the full command `ngrok config add-authtoken ...` (that causes ERR_NGROK_105). → enable **Notebook access** for this notebook.
4. Run the next cell — it should print `OK: NGROK_AUTHTOKEN is set`.

In [ ]:
import os


def _sanitize_ngrok_token(raw: str) -> str:
    """If user pasted full 'ngrok config add-authtoken TOKEN', keep only TOKEN (ERR_NGROK_105)."""
    raw = (raw or "").strip()
    if not raw:
        return ""
    if "add-authtoken" in raw:
        parts = raw.split()
        try:
            i = parts.index("add-authtoken")
            if i + 1 < len(parts):
                return parts[i + 1].strip()
        except ValueError:
            pass
    return raw


def _read_ngrok_token():
    try:
        from google.colab import userdata

        raw = userdata.get("NGROK_AUTHTOKEN").strip()
    except Exception:
        raw = os.environ.get("NGROK_AUTHTOKEN", "").strip()
    return _sanitize_ngrok_token(raw)


tok = _read_ngrok_token()
if tok:
    print("OK: NGROK_AUTHTOKEN is set (length %d)." % len(tok))
else:
    print(
        "MISSING: add Colab Secret NGROK_AUTHTOKEN (key icon) or set env NGROK_AUTHTOKEN.\n"
        "On ngrok.com: verify your email, then copy authtoken from the dashboard."
    )

## 4) Start the API (background) + public URL

Run this once. If you change code, **restart runtime** and re-run from the top.

In [ ]:
import os
import subprocess
import time
import sys


def _sanitize_ngrok_token(raw: str) -> str:
    raw = (raw or "").strip()
    if not raw:
        return ""
    if "add-authtoken" in raw:
        parts = raw.split()
        try:
            i = parts.index("add-authtoken")
            if i + 1 < len(parts):
                return parts[i + 1].strip()
        except ValueError:
            pass
    return raw


def _read_ngrok_token():
    try:
        from google.colab import userdata

        raw = userdata.get("NGROK_AUTHTOKEN").strip()
    except Exception:
        raw = os.environ.get("NGROK_AUTHTOKEN", "").strip()
    return _sanitize_ngrok_token(raw)


TOKEN = _read_ngrok_token()
if not TOKEN:
    raise RuntimeError(
        "ngrok needs an authtoken (ERR_NGROK_4018).\n"
        "1) Create/verify account: https://dashboard.ngrok.com/signup\n"
        "2) Copy authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "3) Colab → Secrets (key) → name NGROK_AUTHTOKEN → paste token → allow notebook access\n"
        "4) Re-run this cell.\n"
        "If this still fails, run the next cell (Cloudflare Tunnel — no ngrok)."
    )

# Debug: token must be non-empty (do NOT print the token itself)
print("Token length:", len(TOKEN), "(real token only; ~40–55 — not the whole shell command)")

from pyngrok import ngrok

# ngrok v3: do NOT pass auth_token into connect() — it breaks the tunnel API (error 102).
ngrok.set_auth_token(TOKEN)
ngrok.kill()

# Stop previous uvicorn if re-running
!pkill -f "uvicorn main:app" 2>/dev/null || true
time.sleep(1)

os.chdir("/content/pharmacie/backend")
sys.path.insert(0, "/content/pharmacie/backend")

log = open("/content/uvicorn.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log,
    stderr=subprocess.STDOUT,
    cwd="/content/pharmacie/backend",
)
time.sleep(3)

tunnel = ngrok.connect(8000)

BASE = getattr(tunnel, "public_url", str(tunnel)).rstrip("/")
print("\n=== Backend base URL (use for Vercel VITE_API_BASE_URL) ===")
print(BASE)
print("\nTest:", BASE + "/api/health")
print("\nKeep this tab open; Colab stops when disconnected.")

## Alternative B — Cloudflare Tunnel (sans ngrok)

Si **ngrok** affiche encore **ERR_NGROK_4018** alors que le secret est bon : exécute la **cellule suivante** (elle installe `cloudflared` et ouvre un lien `https://….trycloudflare.com`).  
Il faut que **uvicorn** tourne déjà sur le port **8000** (cellule du dessus, même si ngrok a planté après le démarrage du serveur).

In [ ]:
import os
import platform
import re
import subprocess
import time
import urllib.request

# Télécharge cloudflared (compatible Colab x86_64 ou arm64)
m = platform.machine().lower()
arch = "arm64" if m in ("aarch64", "arm64") else "amd64"
url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
cf = "/tmp/cloudflared"
print("Téléchargement:", url)
urllib.request.urlretrieve(url, cf)
os.chmod(cf, 0o755)

subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(1)

# Redirige les logs vers un fichier et cherche l'URL HTTPS
log_path = "/tmp/cloudflared.log"
open(log_path, "w").close()
proc = subprocess.Popen(
    [cf, "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=open(log_path, "a"),
    stderr=subprocess.STDOUT,
)
BASE = None
for _ in range(45):
    time.sleep(1)
    if os.path.exists(log_path):
        txt = open(log_path).read()
        m2 = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
        if m2:
            BASE = m2.group(0).rstrip("/")
            break

if not BASE:
    print("Lecture du log (cherche une ligne trycloudflare.com):")
    print(open(log_path).read()[-4000:])
    raise RuntimeError(
        "URL Cloudflare introuvable. Vérifie que uvicorn écoute sur 8000 (cellule précédente)."
    )

print("\n=== BASE (Vercel VITE_API_BASE_URL) ===")
print(BASE)
print("Test:", BASE + "/api/health")

## 5) Quick test (optional)

In [ ]:
import urllib.request
import json

with urllib.request.urlopen(BASE + "/api/health", timeout=30) as r:
    print(json.loads(r.read().decode()))

## Vercel

Set **`VITE_API_BASE_URL`** to the printed **`BASE`** (same as ngrok HTTPS URL), then redeploy the frontend.

---

**If install fails:** open the captured pip output by changing the install cell to remove `%%capture`, re-run, and read errors. Some Colab runtimes need a **Runtime → Restart session** after apt installs.